Performance 🏃 
---
https://github.com/jeffmur/fhe-video-similarity/wiki/experiment

This notebook serves to analyze, compare, and visulize the trade-off of performance when using Fully Homomorphic Encryption (FHE) to compute video similarity scores on mobile vs. desktop devices.

Note: The multi-threading feature for pre-processing is disabled for all experiments, as we value consistency over performance.

Note: Every experiment uses the same encryption scheme & parameters:

* Cryptosystem: CKKS
* Polynomial Degree: 4096
* Encode Scalar: 2^40
* qSizes: [60, 40, 40, 60]

There are three metrics being gather within the application:

⚙️ **Pre-processing Time**: The time it takes to convert the video into a format that can be used for comparison.

📊 **Similarity Scores**: The time it takes to encrypt & compute a similarity score.

# Experiment 1: On-device Comparison

In this experiment, we compare the pre-processing and encryption duration on mobile vs. desktop devices.

We aim to learn how the performance of the application varies amongst devices (mobile vs. desktop). The scenarios are split by resolution (720p, 1080p, 2160p) and the video length is a constant 60 seconds. We select 60 seconds, as there is a direct comparison between the devices, as well as baseline comparison to Pop-Share.

| Device | OS
| --- | --- |
| Samsung S9 | Android
| Pixel 3XL | Android
| PC | Linux
| Raspberry Pi 400 | Linux

In [1]:
from utils.performance_tables import *
from utils import TARGET_SYS, FRAME_COUNTS
# To be summarized in table
kld_err = []
bhatt_err = []
cram_err = []

def add_mean_error(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    kld, bhattacharyya, cramer = mean_error(pathToAssertion, os, frameCounts).values()
    kld_err.append(kld)
    bhatt_err.append(bhattacharyya)
    cram_err.append(cramer)

def insert_or_append(old:dict[str,list], new:dict[str, float]):
    for k, v in new.items():
        if k in old:
            old[k].append(v)
        else:
            old[k] = [v]

# Aggregated Pre-processing durations
pp_by_sys = {}
pp_by_res = {}

def add_metric(pathToAssertion:str, os=TARGET_SYS, frameCounts=FRAME_COUNTS):
    # OS pre-processing
    sys = pre_processing_by_sys(pathToAssertion, os, frameCounts)
    insert_or_append(pp_by_sys, sys)
    
    # Resolution pre-processing
    res = pre_processing_by_res(pathToAssertion.split('/')[1], pathToAssertion, os, frameCounts)
    insert_or_append(pp_by_res, res)
    
    # Mean Error FHE vs. Plaintext
    add_mean_error(pathToAssertion, os, frameCounts)

## Scenario 1: 720p

On every device, import and pre-process the video twice, compare against itself, using two different keys.


### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1280x720 -c:v libx264 -t 60 -an Black_720p_60s.mp4
```

In [2]:
black_720p = "3_performance/720p/60s_Black"
add_metric(black_720p)
verbose_md_table(black_720p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -3.06e-11 [100.00%] | 6.12e-11 | 13.00 | 157.00 | 48.00 | 0.73 | 47.27 [6484.36%]
Cramer [pc] [all] | 1.31e-09 [100.00%] | 2.62e-09 | 13.00 | 71.00 | 113.00 | 0.21 | 112.79 [53201.89%]
BC [pc] [all] | 1.00e+00 [100.00%] | 3.49e-10 | 13.00 | 142.00 | 29.00 | 0.20 | 28.80 [14400.00%]

### Test 2: Samsung S9

Taken from Handheld Samsung S9, 720p, 60 seconds.

In [3]:
s9_720p = "3_performance/720p/60s_S9"
add_metric(s9_720p)
verbose_md_table(s9_720p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -1.97e-12 [100.00%] | 3.94e-12 | 71.00 | 152.00 | 48.00 | 0.61 | 47.39 [7730.34%]
Cramer [pc] [all] | 7.49e-10 [100.00%] | 1.50e-09 | 71.00 | 71.00 | 114.00 | 0.21 | 113.79 [54972.46%]
BC [pc] [all] | 1.00e+00 [100.00%] | 2.42e-10 | 71.00 | 141.00 | 30.00 | 0.16 | 29.84 [18533.54%]

## Scenario 2: 1080p



### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=1920x1080 -c:v libx264 -t 60 -an Black_1080p_60s.mp4
```

In [4]:
black_1080p = "3_performance/1080p/60s_Black"
add_metric(black_1080p)
verbose_md_table(black_1080p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | 3.12e-11 [100.00%] | 6.24e-11 | 36.00 | 148.00 | 45.00 | 0.43 | 44.57 [10463.38%]
Cramer [pc] [all] | 0.00e+00 [100.00%] | 0.00e+00 | 36.00 | 70.00 | 111.00 | 0.11 | 110.89 [99900.00%]
BC [pc] [all] | 1.00e+00 [100.00%] | 4.66e-10 | 36.00 | 142.00 | 27.00 | 0.08 | 26.92 [32430.12%]

### Test 2: Pixel 3XL

Taken from Handheld Pixel 3XL, 1080p, 60 seconds.

In [5]:
pxl_1080p = "3_performance/1080p/60s_PXL"
add_metric(pxl_1080p)
verbose_md_table(pxl_1080p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | 1.56e-11 [100.00%] | 3.12e-11 | 310.00 | 144.00 | 45.00 | 0.37 | 44.63 [12062.16%]
Cramer [pc] [all] | 4.03e-10 [100.00%] | 8.06e-10 | 310.00 | 71.00 | 113.00 | 0.09 | 112.91 [124075.82%]
BC [pc] [all] | 1.00e+00 [100.00%] | 1.12e-10 | 310.00 | 142.00 | 29.00 | 0.09 | 28.91 [32484.27%]

## Scenario 3: 2160p


### Test 1: FFmpeg 60s Black Frames

```bash
ffmpeg -f lavfi -i color=c=black:s=3840x2160 -c:v libx264 -t 60 -an Black_2160p_60s.mp4
```

In [6]:
black_2160p = "3_performance/2160p/60s_Black"
add_metric(black_2160p)
verbose_md_table(black_2160p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -5.81e-12 [100.00%] | 1.16e-11 | 159.00 | 143.00 | 47.00 | 0.32 | 46.68 [14726.50%]
Cramer [pc] [all] | 3.31e-10 [100.00%] | 6.62e-10 | 159.00 | 70.00 | 114.00 | 0.10 | 113.90 [109515.38%]
BC [pc] [all] | 1.00e+00 [100.00%] | 1.97e-10 | 159.00 | 142.00 | 29.00 | 0.09 | 28.91 [33233.33%]

### Test 2: iPhone 13

Taken from Handheld iPhone 13, 2160p, 60 seconds.

In [7]:
iphone_2160p = "3_performance/2160p/60s_iPhone13"
add_metric(iphone_2160p)
verbose_md_table(iphone_2160p)

Similarity [sys] [frameCount] | Score [%] | FHE Absolute Error | Pre-processing (s) | Encryption (ms) | FHE Compute (ms) | Plaintext Compute (ms) | FHE/Plain Compute Growth (ms) [%]
---|---|---|---|---|---|---|---
KLD [pc] [all] | -4.59e-11 [100.00%] | 9.19e-11 | 289.75 | 289.00 | 93.00 | 0.66 | 92.34 [13927.15%]
Cramer [pc] [all] | 1.66e-10 [100.00%] | 3.31e-10 | 289.75 | 142.00 | 229.00 | 0.20 | 228.80 [115556.57%]
BC [pc] [all] | 1.00e+00 [100.00%] | 7.82e-10 | 289.75 | 287.00 | 59.00 | 0.18 | 58.82 [32496.69%]

# Experiment 2: FHE vs. Plaintext Operations

## Scenario 1: Absolute Mean Error

Aggregated from the previous experiments, the absolute mean error will be calculated to determine the accuracy of the similarity scores using FHE library.

In [ ]:
mean_error_md_table(kld_err, bhatt_err, cram_err)

## Scenario 2: Android vs. Linux


# Experiment 4: Pre-processing


## Scenario 1: Android vs. Linux

In [9]:
pp_by_sys

{'pc': [13.0, 71.0, 36.0, 310.0, 159.0, 289.75]}

In [10]:
pre_processing_by_sys_md_table(pp_by_sys)

System | Average (s) | Min (s) | Max (s)
---|---|---|---
pc | 146.46 | 13.00 | 310.00

## Scenario 2: Resolution

In [11]:
pp_by_res

{'720p': [13.0, 71.0], '1080p': [36.0, 310.0], '2160p': [159.0, 289.75]}

In [12]:
pre_processing_by_res_md_table(pp_by_res)

Resolution | Average (s) | Min (s) | Max (s)
---|---|---|---
720p | 42.00 | 13.00 | 71.00
1080p | 173.00 | 36.00 | 310.00
2160p | 224.38 | 159.00 | 289.75